# Stage 0 — Inspect raw data (no cleaning)

This notebook only **looks** at `data/raw/`. It never changes or deletes raw files.
Big files are opened lazily (metadata only, plus at most one time step to count empty values).

The only file it writes is a small list of raw files: `data/interim/raw_file_inventory.csv`.

Run the cells top to bottom. After each cell, check the line "Look for".

## Cell 1 — Setup
Finds the project folder (the one that contains `data/raw`) and checks which reading libraries are installed.

**Look for:** the three folder paths are correct; which libraries say `MISSING`.
`xarray`, `netCDF4` and `openpyxl` are needed. For MODIS `.hdf` files we need `pyhdf` **or** `osgeo` (GDAL) **or** a `netCDF4` that can read HDF4. For `.grib` files we need `cfgrib`.

In [ ]:
from pathlib import Path
import sys, re, zipfile, importlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Project folder = first folder (here or above) that has data/raw inside it
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p / "data" / "raw").is_dir()), None)
assert PROJECT is not None, f"Could not find data/raw in {here} or any parent folder"
RAW = PROJECT / "data" / "raw"
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"

print("Python    :", sys.version.split()[0])
print("Project   :", PROJECT)
print("Raw       :", RAW)
print("Interim   :", INTERIM, "(exists)" if INTERIM.is_dir() else "(MISSING)")
print("Processed :", PROCESSED, "(exists)" if PROCESSED.is_dir() else "(MISSING)")

PERIOD_START, PERIOD_END = pd.Timestamp("2025-06-01"), pd.Timestamp("2026-05-31")

print("\nLibraries:")
for name in ["pandas", "numpy", "xarray", "netCDF4", "h5netcdf", "h5py", "dask",
             "cfgrib", "pyhdf", "osgeo", "rasterio", "openpyxl"]:
    try:
        m = importlib.import_module(name)
        print(f"  OK       {name:10s} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"  MISSING  {name:10s} ({type(e).__name__})")

## Cell 2 — List every raw file
Walks through `data/raw/` and all its subfolders. Prints a summary per folder and per file type, then saves the full list to `data/interim/raw_file_inventory.csv` (small file).

**Look for:** total size close to what you expect (about 29 GB MERRA-2 + 57 GB TROPOMI + 0.5 GB MODIS + 0.1 GB aethalometer + ERA5 + Excel ≈ 87 GB). Folder names and file types.

In [ ]:
rows = []
for p in sorted(RAW.rglob("*")):
    if p.is_file():
        rel = p.relative_to(RAW)
        rows.append({
            "rel_path": rel.as_posix(),
            "folder": rel.parent.as_posix() if rel.parent.as_posix() != "." else "(raw root)",
            "name": p.name,
            "ext": p.suffix.lower() or "(none)",
            "size_mb": p.stat().st_size / 1e6,
        })
inv = pd.DataFrame(rows)
print(f"Total files: {len(inv)}   Total size: {inv.size_mb.sum() / 1e3:,.2f} GB")

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

print("\nPer folder:")
per_folder = inv.groupby("folder").agg(
    files=("name", "size"),
    size_gb=("size_mb", lambda s: round(s.sum() / 1e3, 3)),
    types=("ext", lambda s: ", ".join(sorted(s.unique()))),
    first_file=("name", "first"),
    last_file=("name", "last"),
)
print(per_folder.to_string())

print("\nPer file type:")
print(inv.groupby("ext").agg(files=("name", "size"),
                             size_gb=("size_mb", lambda s: round(s.sum() / 1e3, 3))).to_string())

inv.to_csv(INTERIM / "raw_file_inventory.csv", index=False)
print("\nSaved:", INTERIM / "raw_file_inventory.csv")

## Cell 3 — Guess which dataset each file belongs to
Uses simple keywords in the file name and folder name (for example `merra`, `s5p`/`no2`/`so2`, `mcd19a2`/`modis`, `era5`, `aeth`, `pm2.5`).
It also reads the date written in the file name (if any) and a "sub-group" (for example the MERRA-2 collection name, or NO2 vs SO2).

**Look for:** every file has the right label. The list of `unknown` files at the end. If any label is wrong, tell me — I will fix the keywords. Nothing is moved or renamed.

In [ ]:
# (label, pattern searched in the lower-case path). First match wins.
RULES = [
    ("tropomi", r"s5p|tropomi|l2__no2|l2__so2|no2|so2"),
    ("merra2",  r"merra"),
    ("modis",   r"modis|mcd19a2|m[oy]d04|m[oy]d08|mcd19|(^|[^a-z])aod([^a-z]|$)"),
    ("era5",    r"era5|era-5|reanalysis-era5"),
    ("aeth",    r"aeth|ae33|ae31|ae51|black.?carbon|(^|[^a-z])bc([^a-z]|$)"),
    ("pm25",    r"pm\s*2\.?5|pm25"),
]

def label_of(rel_path):
    s = rel_path.lower()
    for label, pattern in RULES:
        if re.search(pattern, s):
            return label
    return "unknown"

def file_date(name):
    m = re.search(r"\.A(20\d{2})(\d{3})\.", name)              # MODIS style .AYYYYDDD.
    if m:
        return pd.Timestamp(int(m[1]), 1, 1) + pd.Timedelta(days=int(m[2]) - 1)
    m = re.search(r"(20\d{2})[-_]?(\d{2})[-_]?(\d{2})", name)   # YYYYMMDD, YYYY-MM-DD, YYYY_MM_DD
    if m:
        try:
            return pd.Timestamp(int(m[1]), int(m[2]), int(m[3]))
        except ValueError:
            return pd.NaT
    return pd.NaT

SUBGROUP_PATTERNS = {
    "merra2":  r"\.((?:tavg|inst|const|statD)[A-Za-z0-9_]+)\.",   # e.g. tavg1_2d_aer_Nx
    "modis":   r"^([A-Za-z0-9_]+?)\.A20\d{5}",                      # e.g. MCD19A2
    "tropomi": r"L2__([A-Z0-9_]+?)_*20\d{6}T",                      # e.g. NO2, SO2
}

def subgroup_of(label, name):
    pat = SUBGROUP_PATTERNS.get(label)
    m = re.search(pat, name) if pat else None
    return m.group(1) if m else re.sub(r"\d+", "#", name)          # fallback: name pattern

inv["label"] = inv.rel_path.map(label_of)
inv["file_date"] = inv.name.map(file_date)
inv["subgroup"] = [subgroup_of(l, n) for l, n in zip(inv.label, inv.name)]

print("Files per label:")
print(inv.groupby("label").agg(files=("name", "size"),
                               size_gb=("size_mb", lambda s: round(s.sum() / 1e3, 3)),
                               types=("ext", lambda s: ", ".join(sorted(s.unique()))),
                               subgroups=("subgroup", "nunique")).to_string())

print("\nSub-groups per label (first 15 per label):")
for label, g in inv.groupby("label"):
    sg = g.groupby("subgroup").agg(files=("name", "size"), size_gb=("size_mb", lambda s: round(s.sum() / 1e3, 3)))
    print(f"\n[{label}]  {len(sg)} sub-group(s)")
    print(sg.head(15).to_string())

unk = inv[inv.label == "unknown"]
print(f"\nUnknown files: {len(unk)}")
for r in unk.head(40).itertuples():
    print(f"   {r.rel_path}   ({r.size_mb:,.1f} MB)")

inv.to_csv(INTERIM / "raw_file_inventory.csv", index=False)
print("\nUpdated:", INTERIM / "raw_file_inventory.csv")

## Cell 4 — Date coverage from file names (no file is opened)
For each satellite/model sub-group, uses the date in the file names to count days, missing days in 1 Jun 2025 – 31 May 2026, and files per day.

**Look for:** first and last date; number of missing days; files per day (MERRA-2 should be 1 per collection; TROPOMI often 1–3 orbits per day; MODIS may be several tiles/files per day).
If a dataset is stored as monthly or yearly files, "missing days" here means nothing — its dates will come from inside the file (Cells 6–9).

In [ ]:
period_days = pd.date_range(PERIOD_START, PERIOD_END, freq="D")

for (label, sg), g in inv[inv.label.isin(["merra2", "modis", "tropomi", "era5", "aeth"])].groupby(["label", "subgroup"]):
    d = g.file_date.dropna()
    print(f"\n[{label}] {sg}: {len(g)} files, {g.size_mb.sum() / 1e3:,.2f} GB, types {sorted(g.ext.unique())}")
    if d.empty:
        print("   no date in the file names")
        continue
    per_day = d.dt.normalize().value_counts()
    missing = period_days.difference(per_day.index)
    outside = int(((d < PERIOD_START) | (d > PERIOD_END)).sum())
    print(f"   dates in names: {d.min().date()} to {d.max().date()} | {per_day.size} distinct days"
          f" | files per day min/median/max = {per_day.min()}/{per_day.median():.0f}/{per_day.max()}")
    print(f"   days in Jun-2025..May-2026 with no file: {len(missing)}"
          + (f"  first ones: {[x.date().isoformat() for x in missing[:10]]}" if len(missing) else ""))
    print(f"   files dated outside the period: {outside} | files with no date in name: {int(g.file_date.isna().sum())}")

## Cell 5 — Helper functions (prints nothing important)
Defines the tools used by the next cells:
- detect the real file format from its first bytes (not from the extension),
- open NetCDF/HDF5/HDF4/GRIB files **lazily** (metadata only, `cache=False`),
- print dimensions, latitude/longitude range, time range and step, every variable with units, and the share of empty values in **one time step only** (so big files are never fully loaded),
- for text/CSV, Excel and ZIP files: print only the first lines / member list.

GRIB files are opened without writing index files next to the raw data.

**Look for:** the message `Helpers ready.` If it says `xarray not available`, install it before continuing.

In [ ]:
try:
    import xarray as xr
except ImportError:
    xr = None
    print("xarray not available — NetCDF/HDF/GRIB files cannot be inspected")

NC_EXT   = {".nc", ".nc4", ".cdf", ".h5", ".he5", ".hdf5"}
HDF4_EXT = {".hdf", ".hdf4"}
GRIB_EXT = {".grib", ".grb", ".grib2", ".grb2"}
TEXT_EXT = {".csv", ".txt", ".dat", ".tsv", ".json"}
XLS_EXT  = {".xlsx", ".xls", ".xlsm"}

LAT_NAMES  = ["lat", "latitude", "Latitude", "LAT", "XDim_lat", "y"]
LON_NAMES  = ["lon", "longitude", "Longitude", "LON", "x"]
TIME_NAMES = ["time", "valid_time", "Time", "date", "datetime"]
ATTR_KEYS  = r"title|shortname|short_name|version|rangebeginning|rangeending|time_coverage|product|processor|source|institution|conventions|frequency"

def show_size(path):
    path = Path(path)
    print(f"\n=== {path.relative_to(RAW).as_posix()}   ({path.stat().st_size / 1e6:,.1f} MB)")

def magic(path):
    with open(path, "rb") as fh:
        head = fh.read(8)
    if head.startswith(b"\x89HDF"):          return "HDF5 / NetCDF-4"
    if head.startswith(b"CDF"):              return "NetCDF-3 classic"
    if head.startswith(b"\x0e\x03\x13\x01"): return "HDF4"
    if head.startswith(b"GRIB"):             return "GRIB"
    if head.startswith(b"PK"):               return "ZIP (also .xlsx)"
    if head.startswith(b"\xd0\xcf\x11\xe0"): return "old Excel .xls"
    return f"other (first bytes {head!r})"

def list_nc_groups(path):
    try:
        import netCDF4
        out = []
        with netCDF4.Dataset(path) as nc:
            def walk(g, prefix=""):
                for name, sub in g.groups.items():
                    out.append(prefix + name)
                    walk(sub, prefix + name + "/")
            walk(nc)
        return out
    except Exception as e:
        return f"could not list groups ({type(e).__name__}: {str(e)[:120]})"

def open_lazy(path, group=None):
    ext = Path(path).suffix.lower()
    if ext in GRIB_EXT:
        tries = [("cfgrib", {"backend_kwargs": {"indexpath": ""}})]   # no .idx file written into raw
    else:
        tries = [("netcdf4", {}), ("h5netcdf", {})]
    errors = []
    for engine, extra in tries:
        kw = dict(engine=engine, cache=False, mask_and_scale=True, **extra)
        if group:
            kw["group"] = group
        try:
            return xr.open_dataset(path, **kw), engine
        except Exception as e:
            errors.append(f"{engine}: {type(e).__name__}: {str(e)[:150]}")
    print("  could not open with xarray:")
    for e in errors:
        print("    ", e)
    return None, None

def first_name(ds, names):
    return next((n for n in names if n in ds.variables), None)

def print_attrs(attrs):
    keys = [k for k in attrs if re.search(ATTR_KEYS, k, re.I)]
    for k in keys[:12]:
        print(f"  attr {k}: {str(attrs[k])[:120]}")
    sm = attrs.get("StructMetadata.0")
    if sm:   # MODIS HDF-EOS grid info
        print("  HDF-EOS grid:", re.findall(r"(GridName|XDim|YDim|UpperLeftPointMtrs|LowerRightMtrs|Projection)=([^\n]*)", str(sm))[:8])

def describe(ds, first_step_only=True, max_vars=60, max_elems=5e7):
    print("  dims:", dict(ds.sizes))

    # area covered
    la, lo = first_name(ds, LAT_NAMES), first_name(ds, LON_NAMES)
    if not la or not lo:
        print("  no lat/lon variable found (maybe a projected grid, e.g. MODIS sinusoidal)")
    for nm in (la, lo):
        if nm:
            v = ds[nm]
            if v.size <= max_elems:
                vals = np.asarray(v.values, dtype=float)
                step = ""
                if vals.ndim == 1 and vals.size > 1:
                    step = f", step ~{np.nanmedian(np.abs(np.diff(vals))):.4f}"
                print(f"  {nm}: {np.nanmin(vals):.3f} to {np.nanmax(vals):.3f}  ({v.size} values{step})")

    # time range and time step
    tn = first_name(ds, TIME_NAMES)
    if tn:
        t = ds[tn]
        if np.issubdtype(t.dtype, np.datetime64) and t.size <= 1e6:
            tv = pd.DatetimeIndex(np.ravel(t.values)).sort_values()
            steps = pd.Series(tv).diff().dropna().value_counts().head(3)
            print(f"  time ({tn}): {tv.min()} to {tv.max()} | {tv.size} steps | most common steps: "
                  f"{', '.join(f'{s} (x{c})' for s, c in steps.items()) or 'single time'}")
            if t.attrs.get("units") or t.encoding.get("units"):
                print(f"  time units in file: {t.encoding.get('units', t.attrs.get('units'))}")
        else:
            print(f"  time ({tn}): not decoded as dates; dtype={t.dtype}, units={t.attrs.get('units', '?')}")
    else:
        print("  no time variable found")

    # variables, units, empty values
    names = list(ds.data_vars)
    print(f"  {len(names)} data variables" + (f" (showing first {max_vars})" if len(names) > max_vars else ""))
    tdim = tn if (tn and tn in ds.dims) else None
    for name in names[:max_vars]:
        v = ds[name]
        part, note = v, ""
        if first_step_only and tdim and tdim in v.dims:
            part, note = v.isel({tdim: 0}), " (first time step)"
        if v.dtype.kind in "fiuc" and part.size <= max_elems:
            miss = f"{float(part.isnull().mean()) * 100:5.1f}% empty{note}"
        else:
            miss = "empty share not checked"
        desc = str(v.attrs.get("long_name", v.attrs.get("standard_name", "")))[:45]
        print(f"   - {name:30s} dims={v.dims} units={str(v.attrs.get('units', '?'))[:14]:14s} | {desc:45s} | {miss}")

def inspect_hdf4_fallback(path):
    try:
        from pyhdf.SD import SD, SDC
    except ImportError:
        SD = None
    if SD is not None:
        f = SD(str(path), SDC.READ)
        print_attrs(f.attributes())
        print("  HDF4 datasets (read with pyhdf):")
        for name, info in f.datasets().items():
            a = f.select(name).attributes()
            line = (f"   - {name:35s} dims={info[0]} shape={info[1]} units={a.get('units', '?')} "
                    f"scale={a.get('scale_factor', '-')} offset={a.get('add_offset', '-')} fill={a.get('_FillValue', '-')}")
            if re.search(r"optical_depth|aod", name, re.I) and "_FillValue" in a:
                arr = f.select(name).get()
                line += f" | {np.mean(arr == a['_FillValue']) * 100:.1f}% empty"
            print(line)
        f.end()
        return
    try:
        from osgeo import gdal
        g = gdal.Open(str(path))
        print("  HDF4 sub-datasets (read with GDAL):")
        for sub, desc in g.GetSubDatasets():
            print("   -", desc)
        g = None
    except Exception as e:
        print(f"  cannot read HDF4: install pyhdf or GDAL ({type(e).__name__}: {str(e)[:100]})")

def inspect_file(path, first_step_only=True, group="auto"):
    path = Path(path)
    show_size(path)
    ext = path.suffix.lower()
    print("  real format:", magic(path))

    if ext == ".zip":
        with zipfile.ZipFile(path) as z:
            members = z.infolist()
            print(f"  ZIP with {len(members)} files (first 15):")
            for m in members[:15]:
                print(f"    {m.filename}  ({m.file_size / 1e6:,.1f} MB)")
        return

    if ext in TEXT_EXT:
        print("  first 8 lines (raw text):")
        with open(path, "r", errors="replace") as fh:
            for _, line in zip(range(8), fh):
                print("   |", line.rstrip()[:200])
        if path.stat().st_size < 300e6:
            with open(path, "rb") as fh:
                print("  number of lines:", sum(1 for _ in fh))
        return

    if ext in XLS_EXT:
        xls = pd.ExcelFile(path)
        print("  sheets:", xls.sheet_names)
        for sh in xls.sheet_names[:5]:
            print(f"  --- sheet '{sh}', first 6 rows (no header applied):")
            print(pd.read_excel(xls, sheet_name=sh, header=None, nrows=6).to_string(max_cols=15))
        xls.close()
        return

    if ext not in NC_EXT | HDF4_EXT | GRIB_EXT:
        print("  not a known data type — skipped")
        return
    if xr is None:
        print("  xarray missing — skipped")
        return

    g = None
    if ext in NC_EXT:
        groups = list_nc_groups(path)
        print("  groups:", groups if groups else "none")
        if group == "auto" and isinstance(groups, list) and "PRODUCT" in groups:
            g = "PRODUCT"          # TROPOMI keeps the main data in the PRODUCT group
        elif group not in ("auto", None):
            g = group

    ds, engine = open_lazy(path, g)
    if ds is None:
        if ext in HDF4_EXT:
            inspect_hdf4_fallback(path)
        return
    print(f"  opened lazily with {engine}" + (f", group={g}" if g else ""))
    try:
        print_attrs(ds.attrs)
        describe(ds, first_step_only=first_step_only)
    finally:
        ds.close()

def inspect_samples(label, first_step_only=True, files_per_group=1, max_groups=10):
    g = inv[inv.label == label].sort_values("rel_path")
    if g.empty:
        print(f"No files labelled '{label}'. Check the labels in Cell 3.")
        return
    groups = list(g.groupby("subgroup", sort=False))
    if len(groups) > max_groups:
        print(f"{len(groups)} sub-groups; showing the first {max_groups} only")
    for sg, gg in groups[:max_groups]:
        print(f"\n########## {label} | {sg} | {len(gg)} files | first: {gg.name.iloc[0]} | last: {gg.name.iloc[-1]}")
        for r in gg.head(files_per_group).itertuples():
            inspect_file(RAW / r.rel_path, first_step_only=first_step_only)

print("Helpers ready.")

## Cell 6 — MERRA-2 (about 29 GB)
Opens **one** file per MERRA-2 collection (the first one), metadata only. Empty values are counted on the first hour only.

**Look for:** collection names (e.g. `aer`, `slv`), variable names and units (dust/PM in kg m-3, temperature in K…), latitude/longitude range (global or already cut?), time steps per file (24 hourly or 1 daily?), time in UTC.

In [ ]:
inspect_samples("merra2")

## Cell 7 — MODIS AOD (about 500 MB)
Opens **one** file per MODIS product (the first one).

**Look for:** product name (e.g. MCD19A2 / MOD04), the AOD variable name, its scale factor and fill value, the grid (lat/lon or sinusoidal tile like `h24v05`), and how much is empty.
If it says it cannot read HDF4, tell me which libraries were MISSING in Cell 1.

In [ ]:
inspect_samples("modis")

## Cell 8 — TROPOMI NO2 / SO2 (about 57 GB)
Opens **one** file per product (NO2, SO2), only the `PRODUCT` group if it exists. TROPOMI files are single satellite passes, so latitude/longitude are 2-D swaths.

**Look for:** the list of groups, the main column variable name and units (mol m-2), `qa_value`, latitude/longitude range of that pass, and time. If the files are already gridded (Level 3), the layout will look different — that is fine, just paste it.

In [ ]:
inspect_samples("tropomi")

## Cell 9 — ERA5-Land (daily)
ERA5-Land is small, so empty values are counted on the **whole** file. First one file per sub-group in detail, then a quick one-line summary (time range and variables) for **every** ERA5 file.

**Look for:** variable names and units (K, Pa, m, J m-2, m s-1), time range from 2025-06-01 to 2026-05-31 with a 1-day step, latitude/longitude range, and whether the time stamp is 00:00 (UTC day).

In [ ]:
inspect_samples("era5", first_step_only=False)

era5 = inv[inv.label == "era5"].sort_values("rel_path")
if not era5.empty and xr is not None and era5.size_mb.sum() < 5000:
    print("\n\nOne line per ERA5 file:")
    for r in era5.itertuples():
        if r.ext not in NC_EXT | GRIB_EXT:
            print(f"  {r.rel_path}: type {r.ext}, see detail above")
            continue
        ds, _ = open_lazy(RAW / r.rel_path)
        if ds is None:
            continue
        tn = first_name(ds, TIME_NAMES)
        tr = (f"{pd.Timestamp(ds[tn].values.min()).date()} to {pd.Timestamp(ds[tn].values.max()).date()} ({ds[tn].size} steps)"
              if tn and np.issubdtype(ds[tn].dtype, np.datetime64) else "no decoded time")
        print(f"  {r.rel_path} ({r.size_mb:,.1f} MB): {tr} | vars: {list(ds.data_vars)}")
        ds.close()

## Cell 10 — PM2.5 Excel files (hourly, µg/m³)
Reads each sheet **as raw cells** (no header applied) so we can see where the title row and header row are. Prints the first 6 rows, the last 3 rows, cells that mention "UET" in the first 10 rows, and the Python type of the first values in the first 3 columns (to see how dates are stored).

**Look for:** which row holds the header in each file; the exact column name for UET; whether dates are real dates or text like `6/1/2025 1:00`; whether the Jun–Dec 2025 file has the newest hour on top.

In [ ]:
pm = inv[inv.label == "pm25"].sort_values("rel_path")
print(f"{len(pm)} PM2.5 file(s)")

for r in pm.itertuples():
    path = RAW / r.rel_path
    show_size(path)
    if r.ext not in XLS_EXT:
        inspect_file(path)
        continue
    xls = pd.ExcelFile(path)
    print("  sheets:", xls.sheet_names)
    for sh in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sh, header=None)
        print(f"\n  --- sheet '{sh}': {df.shape[0]} rows x {df.shape[1]} columns (raw, no header applied)")
        print("  first 6 rows:")
        print(df.head(6).to_string(max_cols=15))
        print("  last 3 rows:")
        print(df.tail(3).to_string(max_cols=15))
        hits = [(i, j, df.iat[i, j]) for i in range(min(10, len(df))) for j in range(df.shape[1])
                if isinstance(df.iat[i, j], str) and "uet" in df.iat[i, j].lower()]
        print("  cells mentioning 'UET' in the first 10 rows (row, column, text):", hits[:10])
        print("  first 8 values of the first 3 columns, with their Python type:")
        for j in range(min(3, df.shape[1])):
            vals = [f"{v!r} ({type(v).__name__})" for v in df.iloc[:8, j]]
            print(f"    column {j}: {vals}")
    xls.close()

## Cell 11 — Aethalometer black carbon (about 102 MB) — side check only
Shows the first 3 aethalometer files (first lines only). This data will **never** be a model input.

**Look for:** file type, separator, column names (e.g. BC1…BC7 or BC6), time format and time zone, units (ng/m³).

In [ ]:
ae = inv[inv.label == "aeth"].sort_values("rel_path")
print(f"{len(ae)} aethalometer file(s), {ae.size_mb.sum():,.1f} MB, types {sorted(ae.ext.unique())}")
for r in ae.head(3).itertuples():
    inspect_file(RAW / r.rel_path)

## Cell 12 — Files with an unknown label
Shows the first 10 files that Cell 3 could not label, so we can decide what they are.

**Look for:** whether any of them is actually MERRA-2, MODIS, TROPOMI, ERA5, PM2.5 or aethalometer data.

In [ ]:
unk = inv[inv.label == "unknown"].sort_values("rel_path")
print(f"{len(unk)} unknown file(s)")
for r in unk.head(10).itertuples():
    inspect_file(RAW / r.rel_path)

## What to paste back
Paste the full output of **Cells 1, 2, 3 and 4**, and the output of **Cells 6 to 12**.
If one cell prints a very long list, you can cut the middle, but keep the start and the end.
If a cell fails, paste the error and the cell number.